In [8]:
import pandas as pd
data=pd.read_csv('all_stocks_5yr.csv')
data.head()

,date,open,high,low,close,volume,Name
0,2013-02-08,15.07,15.12,14.63,14.75,8407500,AAL
1,2013-02-11,14.89,15.01,14.26,14.46,8882000,AAL
2,2013-02-12,14.45,14.51,14.10,14.27,8126000,AAL
3,2013-02-13,14.30,14.94,14.25,14.66,10259500,AAL
4,2013-02-14,14.94,14.96,13.16,13.99,31879900,AAL


In [10]:
def recommend_stocks(df):
    # 1. 날짜 데이터 형식 변환 및 정렬
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['Name', 'date'])
    
    # 2. 이동평균선 계산 (rolling 함수 활용)
    # window=며칠간의 평균인지를 나타냄
    df['SMA5'] = df.groupby('Name')['close'].transform(lambda x: x.rolling(window=5).mean())
    df['SMA20'] = df.groupby('Name')['close'].transform(lambda x: x.rolling(window=20).mean())
    
    # 3. 매수 신호 포착 (골든크로스: 5일선이 20일선을 돌파할 때)
    df['Signal'] = (df['SMA5'] > df['SMA20']) & (df['SMA5'].shift(1) <= df['SMA20'].shift(1))
    
    # 4. 가장 최근 날짜에 매수 신호가 뜬 종목들만 필터링
    latest_date = df['date'].max()
    recommendations = df[(df['date'] == latest_date) & (df['Signal'] == True)]
    
    return recommendations[['date', 'Name', 'close', 'SMA5', 'SMA20']]


In [11]:
result = recommend_stocks(data)
print(result)

             date Name  close    SMA5    SMA20
543104 2018-02-07  TPR   50.1  47.402  47.1315
